# Projeto de Deep Learning — Polish Companies Bankruptcy

Leitura e organização inicial da base **Polish Companies Bankruptcy** (UCI Machine Learning Repository, id 365).

- Fonte: https://archive.ics.uci.edu/dataset/365/polish+companies+bankruptcy
- DOI: [10.24432/C5F600](https://doi.org/10.24432/C5F600) — Tomczak, S. (2016). Licença CC BY 4.0.
- Artigo introdutório: Zięba, M., Tomczak, S. M., Tomczak, J. M. (2016). *Ensemble boosted trees with synthetic features generation in application to bankruptcy prediction*. Expert Systems with Applications.

## Sobre os dados

Previsão de falência de empresas polonesas. Os dados vêm do EMIS (Emerging Markets Information Service): empresas falidas foram analisadas no período 2000–2012 e empresas ainda em operação em 2007–2013. Cada instância é um demonstrativo financeiro descrito por 64 indicadores (X1–X64) e um rótulo binário `class` (1 = faliu, 0 = não faliu).

São 5 arquivos ARFF. O arquivo `kyear` contém indicadores do *k*-ésimo ano do período de previsão, com o rótulo indicando a situação de falência após (6 − k) anos:

| Arquivo | Dados do | Situação após | Instâncias | Falidas | Não falidas |
|---|---|---|---|---|---|
| `1year.arff` | 1º ano | 5 anos | 7027 | 271 | 6756 |
| `2year.arff` | 2º ano | 4 anos | 10173 | 400 | 9773 |
| `3year.arff` | 3º ano | 3 anos | 10503 | 495 | 10008 |
| `4year.arff` | 4º ano | 2 anos | 9792 | 515 | 9277 |
| `5year.arff` | 5º ano | 1 ano | 5910 | 410 | 5500 |

A base tem valores faltantes (marcados como `?` no ARFF) e é fortemente desbalanceada (~4–7% de falências).

## Pacotes

In [1]:
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd
from scipy.io import arff

pd.set_option('display.max_columns', 70)
pd.set_option('display.max_colwidth', 120)

## Leitura dos arquivos

Os ARFF ficam em `dados/`. Se só o zip estiver lá, ele é extraído automaticamente.

In [2]:
DADOS = Path('dados')
ZIP = DADOS / 'polish+companies+bankruptcy+data.zip'
ARQUIVOS = {k: DADOS / f'{k}year.arff' for k in range(1, 6)}

if not all(p.exists() for p in ARQUIVOS.values()):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DADOS)


def ler_arff(caminho):
    """Lê um ARFF da base; renomeia Attr1..Attr64 -> X1..X64 e converte a classe para int."""
    registros, _ = arff.loadarff(caminho)  # '?' vira NaN
    df = pd.DataFrame(registros)
    df.columns = [c.replace('Attr', 'X') for c in df.columns]
    df['class'] = df['class'].str.decode('utf-8').astype(int)
    return df


bases = {k: ler_arff(p) for k, p in ARQUIVOS.items()}
{k: df.shape for k, df in bases.items()}

{1: (7027, 65), 2: (10173, 65), 3: (10503, 65), 4: (9792, 65), 5: (5910, 65)}

Base empilhada com a coluna `ano` (1–5) indicando de qual arquivo veio cada linha, e `anos_ate_status = 6 - ano`.

> Atenção: os 5 arquivos não são independentes — a mesma empresa pode aparecer em mais de um ano, e a base não traz identificador de empresa. Para modelagem, o usual é trabalhar com cada ano separadamente (`bases[k]`).

In [3]:
df = pd.concat(
    [b.assign(ano=k, anos_ate_status=6 - k) for k, b in bases.items()],
    ignore_index=True,
)
VARS = [f'X{i}' for i in range(1, 65)]

df.shape

(43405, 67)

In [4]:
df.head()

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,X12,X13,X14,X15,X16,X17,X18,X19,X20,X21,X22,X23,X24,X25,X26,X27,X28,X29,X30,X31,X32,X33,X34,X35,X36,X37,X38,X39,X40,X41,X42,X43,X44,X45,X46,X47,X48,X49,X50,X51,X52,X53,X54,X55,X56,X57,X58,X59,X60,X61,X62,X63,X64,class,ano,anos_ate_status
0,0.200550,0.37951,0.39641,2.0472,32.3510,0.38825,0.249760,1.33050,1.1389,0.50494,0.249760,0.65980,0.166600,0.249760,497.42,0.73378,2.6349,0.249760,0.149420,43.370,1.2479,0.21402,0.119980,0.47706,0.50494,0.60411,1.45820,1.7615,5.9443,0.11788,0.149420,94.14,3.8772,0.56393,0.21402,1.7410,593.2700,0.50591,0.128040,0.662950,0.051402,0.128040,114.42,71.050,1.00970,1.52250,49.394,0.185300,0.110850,2.0420,0.37854,0.25792,2.2437,2.2480,348690.0,0.121960,0.39718,0.87804,0.001924,8.4160,5.1372,82.658,4.4158,7.4277,0,1,5
1,0.209120,0.49988,0.47225,1.9447,14.7860,0.00000,0.258340,0.99601,1.6996,0.49788,0.261140,0.51680,0.158350,0.258340,677.96,0.53838,2.0005,0.258340,0.152000,87.981,1.4293,0.24806,0.123040,NaN,0.39542,0.43992,88.44400,16.9460,3.6884,0.26969,0.152000,122.17,2.9876,2.98760,0.20616,1.6996,NaN,0.49788,0.121300,0.086422,0.064371,0.145950,199.49,111.510,0.51045,1.12520,100.130,0.237270,0.139610,1.9447,0.49988,0.33472,17.8660,17.8660,2304.6,0.121300,0.42002,0.85300,0.000000,4.1486,3.2732,107.350,3.4000,60.9870,0,1,5
2,0.248660,0.69592,0.26713,1.5548,-1.1523,0.00000,0.309060,0.43695,1.3090,0.30408,0.312580,0.64184,0.244350,0.309060,794.16,0.45961,1.4369,0.309060,0.236100,73.133,1.4283,0.30260,0.189960,NaN,0.28932,0.37282,86.01100,1.0627,4.3749,0.41929,0.238150,176.93,2.0630,1.42740,0.31565,1.3090,2.3019,0.51537,0.241140,0.322020,0.074020,0.231170,165.51,92.381,0.94807,1.01010,96.372,0.291810,0.222930,1.0758,0.48152,0.48474,1.2098,2.0504,6332.7,0.241140,0.81774,0.76599,0.694840,4.9909,3.9510,134.270,2.7185,5.2078,0,1,5
3,0.081483,0.30734,0.45879,2.4928,51.9520,0.14988,0.092704,1.86610,1.0571,0.57353,0.092704,0.30163,0.094257,0.092704,917.01,0.39803,3.2537,0.092704,0.071428,79.788,1.5069,0.11550,0.062782,0.17193,0.57353,0.36152,0.94076,1.9618,4.6511,0.14343,0.071428,91.37,3.9948,0.37581,0.11550,1.3562,NaN,0.57353,0.088995,0.401390,0.069622,0.088995,180.77,100.980,0.28720,1.56960,84.344,0.085874,0.066165,2.4928,0.30734,0.25033,2.4524,2.4524,20545.0,0.054015,0.14207,0.94598,0.000000,4.5746,3.6147,86.435,4.2228,5.5497,0,1,5
4,0.187320,0.61323,0.22960,1.4063,-7.3128,0.18732,0.187320,0.63070,1.1559,0.38677,0.187320,0.33147,0.121820,0.187320,1133.20,0.32211,1.6307,0.187320,0.115530,57.045,NaN,0.19832,0.115530,0.18732,0.38677,0.32211,1.41380,1.1184,4.1424,0.27884,0.115530,147.04,2.4823,0.32340,0.19832,1.6278,11.2470,0.43489,0.122310,0.293040,0.096680,0.122310,141.62,84.574,0.73919,0.95787,65.936,0.188110,0.116010,1.2959,0.56511,0.40285,1.8839,2.1184,3186.6,0.134850,0.48431,0.86515,0.124440,6.3985,4.3158,127.210,2.8692,7.8980,0,1,5


## Dicionário de variáveis

Definições copiadas da página do UCI (seção *Additional Variable Information*), com tradução livre.

| Variável | Definição (UCI) | Tradução |
|---|---|---|
| X1 | net profit / total assets | lucro líquido / ativo total |
| X2 | total liabilities / total assets | passivo total / ativo total |
| X3 | working capital / total assets | capital de giro / ativo total |
| X4 | current assets / short-term liabilities | ativo circulante / passivo de curto prazo |
| X5 | [(cash + short-term securities + receivables - short-term liabilities) / (operating expenses - depreciation)] * 365 | [(caixa + títulos de curto prazo + recebíveis - passivo de curto prazo) / (despesas operacionais - depreciação)] * 365 |
| X6 | retained earnings / total assets | lucros retidos / ativo total |
| X7 | EBIT / total assets | EBIT / ativo total |
| X8 | book value of equity / total liabilities | valor contábil do patrimônio líquido / passivo total |
| X9 | sales / total assets | vendas / ativo total |
| X10 | equity / total assets | patrimônio líquido / ativo total |
| X11 | (gross profit + extraordinary items + financial expenses) / total assets | (lucro bruto + itens extraordinários + despesas financeiras) / ativo total |
| X12 | gross profit / short-term liabilities | lucro bruto / passivo de curto prazo |
| X13 | (gross profit + depreciation) / sales | (lucro bruto + depreciação) / vendas |
| X14 | (gross profit + interest) / total assets | (lucro bruto + juros) / ativo total |
| X15 | (total liabilities * 365) / (gross profit + depreciation) | (passivo total * 365) / (lucro bruto + depreciação) |
| X16 | (gross profit + depreciation) / total liabilities | (lucro bruto + depreciação) / passivo total |
| X17 | total assets / total liabilities | ativo total / passivo total |
| X18 | gross profit / total assets | lucro bruto / ativo total |
| X19 | gross profit / sales | lucro bruto / vendas |
| X20 | (inventory * 365) / sales | (estoque * 365) / vendas |
| X21 | sales (n) / sales (n-1) | vendas (n) / vendas (n-1) |
| X22 | profit on operating activities / total assets | lucro operacional / ativo total |
| X23 | net profit / sales | lucro líquido / vendas |
| X24 | gross profit (in 3 years) / total assets | lucro bruto (em 3 anos) / ativo total |
| X25 | (equity - share capital) / total assets | (patrimônio líquido - capital social) / ativo total |
| X26 | (net profit + depreciation) / total liabilities | (lucro líquido + depreciação) / passivo total |
| X27 | profit on operating activities / financial expenses | lucro operacional / despesas financeiras |
| X28 | working capital / fixed assets | capital de giro / ativo imobilizado |
| X29 | logarithm of total assets | logaritmo do ativo total |
| X30 | (total liabilities - cash) / sales | (passivo total - caixa) / vendas |
| X31 | (gross profit + interest) / sales | (lucro bruto + juros) / vendas |
| X32 | (current liabilities * 365) / cost of products sold | (passivo circulante * 365) / custo dos produtos vendidos |
| X33 | operating expenses / short-term liabilities | despesas operacionais / passivo de curto prazo |
| X34 | operating expenses / total liabilities | despesas operacionais / passivo total |
| X35 | profit on sales / total assets | lucro sobre vendas / ativo total |
| X36 | total sales / total assets | vendas totais / ativo total |
| X37 | (current assets - inventories) / long-term liabilities | (ativo circulante - estoques) / passivo de longo prazo |
| X38 | constant capital / total assets | capital permanente / ativo total |
| X39 | profit on sales / sales | lucro sobre vendas / vendas |
| X40 | (current assets - inventory - receivables) / short-term liabilities | (ativo circulante - estoque - recebíveis) / passivo de curto prazo |
| X41 | total liabilities / ((profit on operating activities + depreciation) * (12/365)) | passivo total / ((lucro operacional + depreciação) * (12/365)) |
| X42 | profit on operating activities / sales | lucro operacional / vendas |
| X43 | rotation receivables + inventory turnover in days | prazo de recebimento + giro de estoque, em dias |
| X44 | (receivables * 365) / sales | (recebíveis * 365) / vendas |
| X45 | net profit / inventory | lucro líquido / estoque |
| X46 | (current assets - inventory) / short-term liabilities | (ativo circulante - estoque) / passivo de curto prazo |
| X47 | (inventory * 365) / cost of products sold | (estoque * 365) / custo dos produtos vendidos |
| X48 | EBITDA (profit on operating activities - depreciation) / total assets | EBITDA (lucro operacional - depreciação) / ativo total |
| X49 | EBITDA (profit on operating activities - depreciation) / sales | EBITDA (lucro operacional - depreciação) / vendas |
| X50 | current assets / total liabilities | ativo circulante / passivo total |
| X51 | short-term liabilities / total assets | passivo de curto prazo / ativo total |
| X52 | (short-term liabilities * 365) / cost of products sold) | (passivo de curto prazo * 365) / custo dos produtos vendidos |
| X53 | equity / fixed assets | patrimônio líquido / ativo imobilizado |
| X54 | constant capital / fixed assets | capital permanente / ativo imobilizado |
| X55 | working capital | capital de giro (valor absoluto) |
| X56 | (sales - cost of products sold) / sales | (vendas - custo dos produtos vendidos) / vendas |
| X57 | (current assets - inventory - short-term liabilities) / (sales - gross profit - depreciation) | (ativo circulante - estoque - passivo de curto prazo) / (vendas - lucro bruto - depreciação) |
| X58 | total costs /total sales | custos totais / vendas totais |
| X59 | long-term liabilities / equity | passivo de longo prazo / patrimônio líquido |
| X60 | sales / inventory | vendas / estoque |
| X61 | sales / receivables | vendas / recebíveis |
| X62 | (short-term liabilities *365) / sales | (passivo de curto prazo * 365) / vendas |
| X63 | sales / short-term liabilities | vendas / passivo de curto prazo |
| X64 | sales / fixed assets | vendas / ativo imobilizado |
| class | rótulo | 1 = falência, 0 = não falência |

In [5]:
dicionario = pd.DataFrame(
    [
        ('X1', 'net profit / total assets', 'lucro líquido / ativo total'),
        ('X2', 'total liabilities / total assets', 'passivo total / ativo total'),
        ('X3', 'working capital / total assets', 'capital de giro / ativo total'),
        ('X4', 'current assets / short-term liabilities', 'ativo circulante / passivo de curto prazo'),
        ('X5', '[(cash + short-term securities + receivables - short-term liabilities) / (operating expenses - depreciation)] * 365', '[(caixa + títulos de curto prazo + recebíveis - passivo de curto prazo) / (despesas operacionais - depreciação)] * 365'),
        ('X6', 'retained earnings / total assets', 'lucros retidos / ativo total'),
        ('X7', 'EBIT / total assets', 'EBIT / ativo total'),
        ('X8', 'book value of equity / total liabilities', 'valor contábil do patrimônio líquido / passivo total'),
        ('X9', 'sales / total assets', 'vendas / ativo total'),
        ('X10', 'equity / total assets', 'patrimônio líquido / ativo total'),
        ('X11', '(gross profit + extraordinary items + financial expenses) / total assets', '(lucro bruto + itens extraordinários + despesas financeiras) / ativo total'),
        ('X12', 'gross profit / short-term liabilities', 'lucro bruto / passivo de curto prazo'),
        ('X13', '(gross profit + depreciation) / sales', '(lucro bruto + depreciação) / vendas'),
        ('X14', '(gross profit + interest) / total assets', '(lucro bruto + juros) / ativo total'),
        ('X15', '(total liabilities * 365) / (gross profit + depreciation)', '(passivo total * 365) / (lucro bruto + depreciação)'),
        ('X16', '(gross profit + depreciation) / total liabilities', '(lucro bruto + depreciação) / passivo total'),
        ('X17', 'total assets / total liabilities', 'ativo total / passivo total'),
        ('X18', 'gross profit / total assets', 'lucro bruto / ativo total'),
        ('X19', 'gross profit / sales', 'lucro bruto / vendas'),
        ('X20', '(inventory * 365) / sales', '(estoque * 365) / vendas'),
        ('X21', 'sales (n) / sales (n-1)', 'vendas (n) / vendas (n-1)'),
        ('X22', 'profit on operating activities / total assets', 'lucro operacional / ativo total'),
        ('X23', 'net profit / sales', 'lucro líquido / vendas'),
        ('X24', 'gross profit (in 3 years) / total assets', 'lucro bruto (em 3 anos) / ativo total'),
        ('X25', '(equity - share capital) / total assets', '(patrimônio líquido - capital social) / ativo total'),
        ('X26', '(net profit + depreciation) / total liabilities', '(lucro líquido + depreciação) / passivo total'),
        ('X27', 'profit on operating activities / financial expenses', 'lucro operacional / despesas financeiras'),
        ('X28', 'working capital / fixed assets', 'capital de giro / ativo imobilizado'),
        ('X29', 'logarithm of total assets', 'logaritmo do ativo total'),
        ('X30', '(total liabilities - cash) / sales', '(passivo total - caixa) / vendas'),
        ('X31', '(gross profit + interest) / sales', '(lucro bruto + juros) / vendas'),
        ('X32', '(current liabilities * 365) / cost of products sold', '(passivo circulante * 365) / custo dos produtos vendidos'),
        ('X33', 'operating expenses / short-term liabilities', 'despesas operacionais / passivo de curto prazo'),
        ('X34', 'operating expenses / total liabilities', 'despesas operacionais / passivo total'),
        ('X35', 'profit on sales / total assets', 'lucro sobre vendas / ativo total'),
        ('X36', 'total sales / total assets', 'vendas totais / ativo total'),
        ('X37', '(current assets - inventories) / long-term liabilities', '(ativo circulante - estoques) / passivo de longo prazo'),
        ('X38', 'constant capital / total assets', 'capital permanente / ativo total'),
        ('X39', 'profit on sales / sales', 'lucro sobre vendas / vendas'),
        ('X40', '(current assets - inventory - receivables) / short-term liabilities', '(ativo circulante - estoque - recebíveis) / passivo de curto prazo'),
        ('X41', 'total liabilities / ((profit on operating activities + depreciation) * (12/365))', 'passivo total / ((lucro operacional + depreciação) * (12/365))'),
        ('X42', 'profit on operating activities / sales', 'lucro operacional / vendas'),
        ('X43', 'rotation receivables + inventory turnover in days', 'prazo de recebimento + giro de estoque, em dias'),
        ('X44', '(receivables * 365) / sales', '(recebíveis * 365) / vendas'),
        ('X45', 'net profit / inventory', 'lucro líquido / estoque'),
        ('X46', '(current assets - inventory) / short-term liabilities', '(ativo circulante - estoque) / passivo de curto prazo'),
        ('X47', '(inventory * 365) / cost of products sold', '(estoque * 365) / custo dos produtos vendidos'),
        ('X48', 'EBITDA (profit on operating activities - depreciation) / total assets', 'EBITDA (lucro operacional - depreciação) / ativo total'),
        ('X49', 'EBITDA (profit on operating activities - depreciation) / sales', 'EBITDA (lucro operacional - depreciação) / vendas'),
        ('X50', 'current assets / total liabilities', 'ativo circulante / passivo total'),
        ('X51', 'short-term liabilities / total assets', 'passivo de curto prazo / ativo total'),
        ('X52', '(short-term liabilities * 365) / cost of products sold)', '(passivo de curto prazo * 365) / custo dos produtos vendidos'),
        ('X53', 'equity / fixed assets', 'patrimônio líquido / ativo imobilizado'),
        ('X54', 'constant capital / fixed assets', 'capital permanente / ativo imobilizado'),
        ('X55', 'working capital', 'capital de giro (valor absoluto)'),
        ('X56', '(sales - cost of products sold) / sales', '(vendas - custo dos produtos vendidos) / vendas'),
        ('X57', '(current assets - inventory - short-term liabilities) / (sales - gross profit - depreciation)', '(ativo circulante - estoque - passivo de curto prazo) / (vendas - lucro bruto - depreciação)'),
        ('X58', 'total costs /total sales', 'custos totais / vendas totais'),
        ('X59', 'long-term liabilities / equity', 'passivo de longo prazo / patrimônio líquido'),
        ('X60', 'sales / inventory', 'vendas / estoque'),
        ('X61', 'sales / receivables', 'vendas / recebíveis'),
        ('X62', '(short-term liabilities *365) / sales', '(passivo de curto prazo * 365) / vendas'),
        ('X63', 'sales / short-term liabilities', 'vendas / passivo de curto prazo'),
        ('X64', 'sales / fixed assets', 'vendas / ativo imobilizado'),
    ],
    columns=['variavel', 'definicao_uci', 'traducao'],
).set_index('variavel')

dicionario

,definicao_uci,traducao
variavel,,
X1,net profit / total assets,lucro líquido / ativo total
X2,total liabilities / total assets,passivo total / ativo total
X3,working capital / total assets,capital de giro / ativo total
X4,current assets / short-term liabilities,ativo circulante / passivo de curto prazo
X5,[(cash + short-term securities + receivables - short-term liabilities) / (operating expenses - depreciation)] * 365,[(caixa + títulos de curto prazo + recebíveis - passivo de curto prazo) / (despesas operacionais - depreciação)] * 365
...,...,...
X60,sales / inventory,vendas / estoque
X61,sales / receivables,vendas / recebíveis
X62,(short-term liabilities *365) / sales,(passivo de curto prazo * 365) / vendas


In [6]:
# Busca rápida: dicionario.loc['X27'] ou descrever('X27')
def descrever(var):
    return f"{var}: {dicionario.loc[var, 'traducao']}  ({dicionario.loc[var, 'definicao_uci']})"

descrever('X27')

'X27: lucro operacional / despesas financeiras  (profit on operating activities / financial expenses)'

## Checagens iniciais

In [7]:
# Contagem por ano e proporção de falências (compare com a tabela do UCI)
resumo = df.groupby('ano')['class'].agg(instancias='size', falidas='sum')
resumo['nao_falidas'] = resumo['instancias'] - resumo['falidas']
resumo['prop_falidas'] = (100*resumo['falidas'] / resumo['instancias']).round(2)
resumo

,instancias,falidas,nao_falidas,prop_falidas
ano,,,,
1,7027,271,6756,3.86
2,10173,400,9773,3.93
3,10503,495,10008,4.71
4,9792,515,9277,5.26
5,5910,410,5500,6.94


In [8]:
# Valores faltantes por variável e por ano (% de linhas)
faltantes = df.groupby('ano')[VARS].apply(lambda g: g.isna().mean() * 100).T.round(2)
faltantes['geral'] = (df[VARS].isna().mean() * 100).round(2)
faltantes.sort_values('geral', ascending=False).head(15)

ano,1,2,3,4,5,geral
X37,38.99,44.41,45.09,45.36,43.11,43.74
X21,23.08,31.10,7.68,1.61,1.74,13.49
X27,4.43,6.94,6.81,6.55,6.62,6.37
X60,1.92,5.34,5.64,6.27,4.53,4.96
X45,1.91,5.32,5.63,6.26,4.53,4.95
X24,1.76,2.21,2.16,2.15,2.28,2.12
X28,0.48,2.08,2.17,2.36,1.81,1.87
X64,0.48,2.08,2.17,2.36,1.81,1.87
X53,0.48,2.08,2.17,2.36,1.81,1.87
X54,0.48,2.08,2.17,2.36,1.81,1.87


In [9]:
# Linhas com pelo menos um faltante, por ano
df.assign(tem_na=df[VARS].isna().any(axis=1)).groupby('ano')['tem_na'].mean().round(3)

ano
1    0.545
2    0.598
3    0.535
4    0.513
5    0.487
Name: tem_na, dtype: float64

In [10]:
# Os indicadores são razões com escalas muito diferentes e caudas pesadas
# (relevante para padronização antes da rede neural)
df[VARS].describe(percentiles=[.01, .5, .99]).T

,count,mean,std,min,1%,50%,99%,max
X1,43397.0,0.035160,2.994109,-4.638900e+02,-0.576798,0.04966,0.605991,9.428000e+01
X2,43397.0,0.590212,5.842748,-4.308700e+02,0.018522,0.47190,2.056324,4.809600e+02
X3,43397.0,0.114431,5.439429,-4.799600e+02,-1.000640,0.19661,0.889821,2.833600e+01
X4,43271.0,6.314702,295.434425,-4.031100e-01,0.199897,1.56980,32.468500,5.343300e+04
X5,43316.0,-385.346602,61243.025874,-1.190300e+07,-2978.500000,-1.03450,2879.175000,1.250100e+06
...,...,...,...,...,...,...,...,...
X60,41253.0,448.085823,32345.600072,-1.244000e+01,1.246628,9.79170,815.695600,4.818700e+06
X61,43303.0,17.033202,553.049406,-1.265600e+01,0.610161,6.63630,97.639800,1.080000e+05
X62,43278.0,1502.327833,139266.699576,-2.336500e+06,4.933659,71.32600,915.715500,2.501600e+07
X63,43271.0,9.343074,124.177354,-1.543200e+00,0.293002,5.08760,57.916300,2.345400e+04
